# 07 — Export web artifacts

This notebook produces every number the showcase site and the Risk Console display.
It loads the four exported model artifacts **read-only**, scores the held-out test sets,
computes SHAP explanations for a stratified sample of cases, and writes compact JSON
files to `results/web/`.

Rules this notebook follows:

- No model or preprocessor is fitted, refitted or re-saved. Only `.transform()`,
  `.predict_proba()` and attribute reads are used. The SHA-256 of each `.pkl` is printed
  before and after so you can see the files were not touched.
- Scoring uses the exact test files each model was evaluated on in its notebook:
  financial from the Week 5 processed split, operations from the Week 6 corrected
  27-feature (`_v2`) split.
- Every exported SHAP case is checked: base value plus the sum of **all** feature
  contributions must equal the model's probability to within 1e-6. The notebook stops if
  any case fails.

Inputs that are too large or not redistributable (raw datasets, processed CSVs, the 1.24 GB
Random Forest) are read from the course project folder, set with the `AIT506_DIR`
environment variable. See `data/README.md` for how to get them.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm notice from shap's import; not relevant here
import os, json, hashlib, platform
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import shap
import sklearn
import xgboost
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = Path(os.environ.get("AIT506_DIR", REPO / "data" / "raw" / "AIT506"))
RF_PKL = Path(os.environ.get("CEROP_RF_PKL", SRC / "CEROP_Live_Demo" / "models" / "cerop_operations_rf.pkl"))
OUT = REPO / "results" / "web"
OUT.mkdir(parents=True, exist_ok=True)

print("python", platform.python_version(), "| xgboost", xgboost.__version__,
      "| scikit-learn", sklearn.__version__, "| shap", shap.__version__, "| pandas", pd.__version__)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

ARTIFACTS = {
    "financial_xgb": REPO / "models" / "cerop_financial_xgb.pkl",
    "financial_preprocessor": REPO / "models" / "cerop_financial_preprocessor.pkl",
    "operations_preprocessor": REPO / "models" / "cerop_operations_preprocessor.pkl",
    "operations_rf": RF_PKL,
}
HASH_BEFORE = {k: sha256(p) for k, p in ARTIFACTS.items()}
for k, h in HASH_BEFORE.items():
    print(f"{k:<24} sha256 {h}")

python 3.12.10 | xgboost 3.4.1 | scikit-learn 1.9.0 | shap 0.52.0 | pandas 3.0.3


financial_xgb            sha256 5769f46c80324094f6569d2b2d9c99d68dcb91ba70afc34d42e6789d14a2ed7c
financial_preprocessor   sha256 92c6239e7dfa0a6d43599437d1d36b8cb8f75b7595ef1e93ec4c548b68908727
operations_preprocessor  sha256 903708ae9088c1dbdf554b152fd502f18b0d6a37ea5020d2e2413a59b9938b3f
operations_rf            sha256 b190d51974aca308bbd022f0fbe588df120cc37ca47522e1043245a5d59f6390


## 1. Load the exported artifacts (read-only)

In [2]:
xgb_model = joblib.load(ARTIFACTS["financial_xgb"])
fin_pre = joblib.load(ARTIFACTS["financial_preprocessor"])
ops_pre = joblib.load(ARTIFACTS["operations_preprocessor"])
rf_model = joblib.load(ARTIFACTS["operations_rf"])

FIN_FEATURES = list(xgb_model.feature_names_in_)
OPS_FEATURES = list(rf_model.feature_names_in_)
print("financial model:", type(xgb_model).__name__, len(FIN_FEATURES), "features")
print("operations model:", type(rf_model).__name__, len(OPS_FEATURES), "features,",
      rf_model.n_estimators, "trees")

financial model: XGBClassifier 85 features
operations model: RandomForestClassifier 27 features, 500 trees


## 2. Financial layer: score all 1,364 test firms

In [3]:
fin_dir = SRC / "Week 5" / "GROUP" / "Ankit" / "data"
X_fin = pd.read_csv(fin_dir / "financial_X_test_processed.csv")[FIN_FEATURES]
y_fin = pd.read_csv(fin_dir / "financial_y_test.csv").iloc[:, 0].to_numpy()
X_fin_train = pd.read_csv(fin_dir / "financial_X_train_processed.csv")[FIN_FEATURES]
y_fin_train = pd.read_csv(fin_dir / "financial_y_train.csv").iloc[:, 0].to_numpy()

# Raw (unscaled) values, for display only. Confirm they line up row for row with the
# processed file by running them through the fitted scaler with .transform().
unscaled_dir = SRC / "Week 4" / "Final" / "Outputs"
X_fin_raw = pd.read_csv(unscaled_dir / "financial_X_test_clean_unscaled.csv")[list(fin_pre.feature_names_in_)]
X_fin_train_raw = pd.read_csv(unscaled_dir / "financial_X_train_clean_unscaled.csv")[list(fin_pre.feature_names_in_)]
align = np.abs(fin_pre.transform(X_fin_raw) - X_fin[list(fin_pre.get_feature_names_out())].to_numpy()).max()
print("raw -> scaler.transform vs processed test file, max abs diff:", align)
assert align < 1e-9

fin_proba = xgb_model.predict_proba(X_fin)[:, 1]

def metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {"threshold": thr, "accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred),
            "recall": recall_score(y, pred), "f1": f1_score(y, pred), "roc_auc": roc_auc_score(y, p),
            "pr_auc": average_precision_score(y, p), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

fin_m = metrics(y_fin, fin_proba, 0.50)
print(pd.Series(fin_m).to_string())

raw -> scaler.transform vs processed test file, max abs diff:

 1.794120407794253e-13
threshold       0.500000
accuracy        0.969208
precision       0.521739
recall          0.545455
f1              0.533333
roc_auc         0.956250
pr_auc          0.483788
tn           1298.000000
fp             22.000000
fn             20.000000
tp             24.000000


The drift note in the project log says this exported XGBoost gives F1 ≈ 0.533 on this machine,
against the officially reported 0.529 from the team's Colab run. Check both against files.

In [4]:
w6_fin = pd.read_csv(REPO / "results" / "week6_financial_tuning_results.csv")
official = w6_fin.loc[w6_fin["Model"] == "XGBoost - Week 5 published"].iloc[0]
w5_rerun = pd.read_csv(REPO / "results" / "week5_financial_model_comparison.csv").set_index("Model").loc["XGBoost"]
print(f"official (Colab run)       F1 = {official['F1-Score']:.4f}")
print(f"week5 CSV (local re-run)   F1 = {w5_rerun['F1-Score']:.4f}")
print(f"exported model, here       F1 = {fin_m['f1']:.4f}")
assert abs(fin_m["f1"] - 0.5333) < 1e-3, "exported XGBoost no longer reproduces the documented drift value"
assert abs(fin_m["f1"] - w5_rerun["F1-Score"]) < 1e-9

with open(OUT / "financial_test_scores.json", "w") as f:
    json.dump({
        "model": "XGBoost (cerop_financial_xgb.pkl)",
        "threshold": 0.50,
        "n": int(len(y_fin)), "positives": int(y_fin.sum()),
        "metrics_reproduced": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in fin_m.items()},
        "metrics_official": {"source": "results/week6_financial_tuning_results.csv, row 'XGBoost - Week 5 published'",
                             "precision": round(float(official["Precision"]), 6), "recall": round(float(official["Recall"]), 6),
                             "f1": round(float(official["F1-Score"]), 6), "roc_auc": round(float(official["ROC-AUC"]), 6),
                             "pr_auc": round(float(official["PR-AUC"]), 6)},
        "proba": [round(float(p), 6) for p in fin_proba],
        "label": [int(v) for v in y_fin],
    }, f, separators=(",", ":"))
print("wrote financial_test_scores.json")

official (Colab run)       F1 = 0.5287
week5 CSV (local re-run)   F1 = 0.5333
exported model, here       F1 = 0.5333
wrote financial_test_scores.json


## 3. Operations layer: score all 34,393 test orders

The Random Forest was trained on the Week 6 corrected dataset, which drops the 8 leaky
`Order Status_*` columns. Scoring uses that same `_v2` test file, reindexed to
`model.feature_names_in_`.

In [5]:
ops_dir = SRC / "Week 6" / "Group" / "data"
X_ops = pd.read_csv(ops_dir / "operations_X_test_processed_v2.csv").reindex(columns=OPS_FEATURES)
y_ops = pd.read_csv(ops_dir / "operations_y_test_v2.csv").iloc[:, 0].to_numpy()
y_ops_train = pd.read_csv(ops_dir / "operations_y_train_v2.csv").iloc[:, 0].to_numpy()
assert X_ops.notna().all().all()

ops_proba = rf_model.predict_proba(X_ops)[:, 1]

w6_ops = pd.read_csv(REPO / "results" / "week6_operations_tuning_results.csv")
w6_cm = pd.read_csv(REPO / "results" / "week6_operations_confusion_matrices.csv")
cal = w6_ops.loc[w6_ops["Model"] == "Tuned RF at Calibrated Threshold"].iloc[0]
OPS_THRESHOLD = float(cal["Threshold"])
print("calibrated threshold from week6_operations_tuning_results.csv:", OPS_THRESHOLD)

ops_cal = metrics(y_ops, ops_proba, OPS_THRESHOLD)
ops_def = metrics(y_ops, ops_proba, 0.50)
for k_csv, k in [("Precision", "precision"), ("Recall", "recall"), ("F1", "f1"), ("ROC-AUC", "roc_auc"), ("PR-AUC", "pr_auc"), ("Accuracy", "accuracy")]:
    d = abs(float(cal[k_csv]) - ops_cal[k])
    print(f"{k:<9} reported {float(cal[k_csv]):.4f}  recomputed {ops_cal[k]:.4f}  diff {d:.1e}")
    assert d <= 1e-3, f"{k} differs from Week 6 result by more than 0.001, stopping"

for name, m in [("Tuned RF at Calibrated Threshold", ops_cal), ("Tuned RF at 0.50", ops_def)]:
    row = w6_cm.loc[w6_cm["Model"] == name].iloc[0]
    got = (m["tn"], m["fp"], m["fn"], m["tp"])
    want = tuple(int(row[c]) for c in ["TN", "FP", "FN", "TP"])
    print(f"{name:<34} confusion {got}  week6 csv {want}")
    assert got == want

calibrated threshold from week6_operations_tuning_results.csv: 0.4
precision reported 0.7095  recomputed 0.7095  diff 0.0e+00
recall    reported 0.8254  recomputed 0.8254  diff 0.0e+00
f1        reported 0.7631  recomputed 0.7631  diff 0.0e+00
roc_auc   reported 0.8258  recomputed 0.8258  diff 8.5e-09
pr_auc    reported 0.8730  recomputed 0.8730  diff 1.1e-08
accuracy  reported 0.7191  recomputed 0.7191  diff 0.0e+00
Tuned RF at Calibrated Threshold   confusion (9168, 6371, 3291, 15563)  week6 csv (9168, 6371, 3291, 15563)
Tuned RF at 0.50                   confusion (13368, 2171, 6996, 11858)  week6 csv (13368, 2171, 6996, 11858)


Probabilities are exported at full double precision. Rounding was tried first (4 decimals, then 6)
and both moved a few orders across a slider step, because some Random Forest averages land a hair
below a round number (for example 0.39999999999999997). JSON keeps the exact double, and JavaScript
compares it with the same double the slider produces, so the explorer matches numpy on every step.
The check below proves that by re-reading the written values.

In [6]:
ops_export = [float(p) for p in ops_proba]
grid = np.round(np.arange(0.01, 1.00, 0.01), 2)
with open(OUT / "operations_test_scores.json", "w") as f:
    json.dump({
        "model": "Random Forest (cerop_operations_rf.pkl, Week 6 tuned final_model)",
        "threshold_calibrated": OPS_THRESHOLD, "threshold_default": 0.50,
        "n": int(len(y_ops)), "positives": int(y_ops.sum()),
        "metrics_calibrated": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in ops_cal.items()},
        "metrics_default": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in ops_def.items()},
        "proba": ops_export,
        "label": [int(v) for v in y_ops],
    }, f, separators=(",", ":"))
reread = np.array(json.load(open(OUT / "operations_test_scores.json"))["proba"])
flips = sum(int(((reread >= t) != (ops_proba >= t)).sum()) for t in grid)
print("wrote operations_test_scores.json; slider steps checked:", len(grid), "| rows that change side:", flips)
assert flips == 0 and np.array_equal(reread, ops_proba)

wrote operations_test_scores.json; slider steps checked: 99 | rows that change side: 0


## 4. Dataset facts used on the site

Counted from the label files, not typed in.

In [7]:
facts = {
    "financial": {"firms": int(len(y_fin_train) + len(y_fin)), "train": int(len(y_fin_train)), "test": int(len(y_fin)),
                  "bankrupt": int(y_fin_train.sum() + y_fin.sum()), "test_bankrupt": int(y_fin.sum()),
                  "bankrupt_rate": round(float((y_fin_train.sum() + y_fin.sum()) / (len(y_fin_train) + len(y_fin))), 6),
                  "features": len(FIN_FEATURES)},
    "operations": {"orders": int(len(y_ops_train) + len(y_ops)), "train": int(len(y_ops_train)), "test": int(len(y_ops)),
                   "late": int(y_ops_train.sum() + y_ops.sum()), "test_late": int(y_ops.sum()),
                   "late_rate": round(float((y_ops_train.sum() + y_ops.sum()) / (len(y_ops_train) + len(y_ops))), 6),
                   "processed_features": int(len(ops_pre.get_feature_names_out())), "model_features": len(OPS_FEATURES)},
}
assert facts["operations"]["orders"] == 137569 + 34393
print(json.dumps(facts, indent=2))
with open(OUT / "dataset_facts.json", "w") as f:
    json.dump(facts, f, indent=1)

{
  "financial": {
    "firms": 6819,
    "train": 5455,
    "test": 1364,
    "bankrupt": 220,
    "test_bankrupt": 44,
    "bankrupt_rate": 0.032263,
    "features": 85
  },
  "operations": {
    "orders": 171962,
    "train": 137569,
    "test": 34393,
    "late": 94268,
    "test_late": 18854,
    "late_rate": 0.548191,
    "processed_features": 35,
    "model_features": 27
  }
}


## 5. Raw operations values for display

The canonical pipeline never saved the raw (unencoded) operations test split. It is rebuilt
here by replaying the Week 4 integration notebook's cleaning steps (date-tail filter, split with
`random_state=42`, structural cleaning, IQR capping, GSCPI merge), the same way
`scripts/regenerate_operations_raw.py` does, from a local copy of the DataCo CSV.

These raw values are used **only for labels on screen** (for example "Shipping Mode (First Class)").
Scores and SHAP values come from the `_v2` processed file the model was evaluated on.

In [8]:
def rebuild_operations_raw(dataco_csv, gscpi_xls):
    df = pd.read_csv(dataco_csv, encoding="latin-1")
    df.columns = df.columns.str.strip()
    df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"], errors="coerce")
    df = df[df["order date (DateOrders)"] < "2017-10-01"].copy()
    from sklearn.model_selection import train_test_split
    tr, te = train_test_split(df, test_size=0.20, stratify=df["Late_delivery_risk"], random_state=42)
    drop_cols = ["Delivery Status", "Days for shipping (real)", "shipping date (DateOrders)", "Customer Email",
                 "Customer Fname", "Customer Lname", "Customer Password", "Customer Street", "Product Image",
                 "Product Description", "Order Customer Id", "Order Item Cardprod Id", "Product Category Id",
                 "Product Status", "Latitude", "Longitude", "Order Id", "Order Item Id", "Customer Zipcode",
                 "Order Zipcode", "Customer City", "Order City", "Order State", "Product Name", "Customer Id",
                 "Category Id", "Department Id", "Product Card Id", "Order Item Total", "Order Item Product Price",
                 "Order Profit Per Order", "Sales per customer", "order date (DateOrders)"]
    def clean(d):
        d = d.copy()
        dt = d["order date (DateOrders)"]
        d["Order_Year"], d["Order_Month"] = dt.dt.year, dt.dt.month
        d["Order_DayOfWeek"], d["Order_Hour"] = dt.dt.dayofweek, dt.dt.hour
        d["Order_YearMonth"] = dt.dt.to_period("M").dt.to_timestamp()
        return d.drop(columns=[c for c in drop_cols if c in d.columns])
    tr, te = clean(tr), clean(te)
    q1, q3 = tr["Benefit per order"].quantile([0.25, 0.75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    tr["Benefit per order"] = tr["Benefit per order"].clip(lo, hi)
    te["Benefit per order"] = te["Benefit per order"].clip(lo, hi)
    g = pd.read_excel(gscpi_xls)
    if "Date" not in g.columns or "GSCPI" not in g.columns:
        g = pd.read_excel(gscpi_xls, skiprows=4)
    g = g.iloc[:, :2].copy(); g.columns = ["Date", "GSCPI"]
    g["Order_YearMonth"] = pd.to_datetime(g["Date"], errors="coerce").dt.to_period("M").dt.to_timestamp()
    g = g[["Order_YearMonth", "GSCPI"]].dropna(subset=["Order_YearMonth"]).drop_duplicates(subset=["Order_YearMonth"])
    tr = tr.merge(g, on="Order_YearMonth", how="left")
    te = te.merge(g, on="Order_YearMonth", how="left")
    return (tr.drop(columns=["Late_delivery_risk", "Order_YearMonth"]), te.drop(columns=["Late_delivery_risk", "Order_YearMonth"]),
            te["Late_delivery_risk"].to_numpy())

X_ops_raw_train, X_ops_raw, y_ops_raw = rebuild_operations_raw(
    SRC / "Week 3" / "Dataset" / "DataCoSupplyChainDataset.csv", SRC / "Week 4" / "Final" / "gscpi_data.xls")
assert (y_ops_raw == y_ops).all(), "rebuilt labels do not line up with the _v2 test labels"

# Row alignment check: transform the rebuilt raw rows with the exported preprocessor and compare
# with the _v2 file the model was scored on.
T = pd.DataFrame(ops_pre.transform(X_ops_raw), columns=ops_pre.get_feature_names_out())[OPS_FEATURES]
diff = (T - X_ops).abs().max()
print(diff[diff > 1e-9])
assert set(diff[diff > 1e-9].index) == {"numeric__Order Item Profit Ratio"}

numeric__Order Item Profit Ratio    4.026127
dtype: float64


Every column matches exactly except `Order Item Profit Ratio`. The `_v2` data used for Week 6
caps that column at the training IQR bounds and standard-scales it on the capped training values.
The exported Week 4 preprocessor does neither for this column. The check below confirms that
exact relationship for all 34,393 rows. It matters for the 8-case live demo script, which chains
the Week 4 preprocessor into the Week 6 model (see notebook 06 and `docs/engineering_log.md`).

In [9]:
pr_tr = X_ops_raw_train["Order Item Profit Ratio"]
q1, q3 = pr_tr.quantile([0.25, 0.75]); pr_lo, pr_hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
capped_tr = pr_tr.clip(pr_lo, pr_hi)
pr_mean, pr_std = capped_tr.mean(), capped_tr.std(ddof=0)
v2_pr = (X_ops_raw["Order Item Profit Ratio"].clip(pr_lo, pr_hi) - pr_mean) / pr_std
resid = np.abs(v2_pr.to_numpy() - X_ops["numeric__Order Item Profit Ratio"].to_numpy()).max()
print(f"Profit Ratio: IQR bounds [{pr_lo:.4f}, {pr_hi:.4f}], capped mean {pr_mean:.6f}, std {pr_std:.6f}, max residual {resid:.1e}")
assert resid < 1e-9
with open(OUT / "operations_profit_ratio_v2_transform.json", "w") as f:
    json.dump({"iqr_low": pr_lo, "iqr_high": pr_hi, "capped_train_mean": pr_mean, "capped_train_std": pr_std,
               "max_residual_vs_v2_test": resid}, f, indent=1)

Profit Ratio: IQR bounds [-0.3400, 0.7800], capped mean 0.191246, std 0.251088, max residual 4.4e-16


## 6. SHAP explanations, financial layer (XGBoost)

Method, unchanged from `scripts/run_shap_explanations.py`: interventional TreeSHAP in probability
space with a 100-row background sample from the training set. The `enable_categorical=True` flag
on the saved model is a library-version default (none of the 85 features are categorical), and
shap refuses it on this path. It is switched off **in memory only**, after checking it does not
change a single predicted probability.

In [10]:
p_before = xgb_model.predict_proba(X_fin)[:, 1]
xgb_model.enable_categorical = False
p_after = xgb_model.predict_proba(X_fin)[:, 1]
assert np.array_equal(p_before, p_after)
print("predict_proba identical with enable_categorical False:", np.array_equal(p_before, p_after))

fin_background = pd.read_csv(fin_dir / "financial_X_train_processed.csv").sample(n=100, random_state=42)[FIN_FEATURES]
fin_explainer = shap.TreeExplainer(xgb_model, data=fin_background, feature_perturbation="interventional",
                                   model_output="probability")
fin_base = float(fin_explainer.expected_value)
fin_shap_all = fin_explainer.shap_values(X_fin)
recon = np.abs(fin_base + fin_shap_all.sum(axis=1) - fin_proba)
print(f"base value {fin_base:.6f}; reconciliation over all 1,364 test firms: max diff {recon.max():.2e}")
assert recon.max() <= 1e-6

fin_importance = pd.Series(np.abs(fin_shap_all).mean(axis=0), index=FIN_FEATURES).sort_values(ascending=False)
print(fin_importance.head(10).round(5).to_string())

predict_proba identical with enable_categorical False: True


base value 0.032400; reconciliation over all 1,364 test firms: max diff 3.88e-07
Quick Ratio                                    0.00838
Total debt/Total net worth                     0.00752
Persistent EPS in the Last Four Seasons        0.00528
Retained Earnings to Total Assets              0.00495
Profitability_Composite                        0.00411
Borrowing dependency                           0.00323
Continuous interest rate (after tax)           0.00299
Non-industry income and expenditure/revenue    0.00285
Fixed Assets to Assets                         0.00239
Allocation rate per person                     0.00239


In [11]:
def cell_of(y, pred):
    return np.where((y == 1) & (pred == 1), "TP", np.where((y == 0) & (pred == 1), "FP",
                    np.where((y == 1) & (pred == 0), "FN", "TN")))

rng = np.random.default_rng(42)
fin_cells = cell_of(y_fin, (fin_proba >= 0.50).astype(int))
FIN_PICK = {"TP": 8, "FN": 6, "FP": 4, "TN": 6}
fin_idx = np.concatenate([np.sort(rng.choice(np.where(fin_cells == c)[0], size=min(n, (fin_cells == c).sum()), replace=False))
                          for c, n in FIN_PICK.items()])

FIN_LABELS = {
    "Total debt/Total net worth": "Total debt / net worth",
    "Retained Earnings to Total Assets": "Retained earnings / assets",
    "Persistent EPS in the Last Four Seasons": "EPS, last four seasons",
    "Total expense/Assets": "Total expense / assets",
    "Current Liabilities/Liability": "Current liabilities / liability",
    "Continuous Net Profit Growth Rate": "Net profit growth rate",
    "Revenue Per Share (Yuan ¥)": "Revenue per share",
    "Net Value Growth Rate": "Net value growth rate",
    "Quick Ratio": "Quick ratio",
    "Profitability_Composite": "Profitability (average of 3 ROA measures)",
    "Cash/Total Assets": "Cash / total assets",
    "Fixed Assets to Assets": "Fixed assets / assets",
    "Continuous interest rate (after tax)": "Interest rate (after tax)",
    "Non-industry income and expenditure/revenue": "Non-industry income / revenue",
}
TOP_N = 6

def band(p, thr):
    return "High" if p >= thr else ("Watch" if p >= thr / 2 else "Clear")

fin_cases = []
for i in fin_idx:
    contrib = sorted(zip(FIN_FEATURES, fin_shap_all[i]), key=lambda t: abs(t[1]), reverse=True)
    top = contrib[:TOP_N]
    rest = float(fin_proba[i] - fin_base - sum(v for _, v in top))
    total = fin_base + sum(v for _, v in contrib)
    assert abs(total - fin_proba[i]) <= 1e-6
    fin_cases.append({
        "id": f"FIN-T{i:04d}", "row": int(i), "cell": str(fin_cells[i]), "label": int(y_fin[i]),
        "proba": round(float(fin_proba[i]), 6), "band": band(fin_proba[i], 0.50),
        "drivers": [{"feature": n, "label": FIN_LABELS.get(n, n), "contribution": round(float(v), 6),
                     "raw_value": round(float(X_fin_raw.iloc[i][n]), 6)} for n, v in top],
        "rest": {"label": f"{len(FIN_FEATURES) - TOP_N} more factors combined", "contribution": round(rest, 6)},
        "reconciliation_abs_diff": float(abs(total - fin_proba[i])),
    })

with open(OUT / "financial_shap_cases.json", "w") as f:
    json.dump({"method": "shap.TreeExplainer, interventional, model_output='probability', 100-row training background (random_state=42)",
               "base_value": round(fin_base, 6), "threshold": 0.50, "top_n": TOP_N,
               "global_mean_abs_shap": [{"feature": n, "label": FIN_LABELS.get(n, n), "value": round(float(v), 6)}
                                        for n, v in fin_importance.head(15).items()],
               "cases": fin_cases}, f, indent=1)
print(f"wrote financial_shap_cases.json: {len(fin_cases)} cases,", pd.Series([c["cell"] for c in fin_cases]).value_counts().to_dict())
print("max reconciliation diff across exported financial cases:", max(c["reconciliation_abs_diff"] for c in fin_cases))

wrote financial_shap_cases.json: 24 cases, {'TP': 8, 'FN': 6, 'TN': 6, 'FP': 4}
max reconciliation diff across exported financial cases: 3.8789515943715713e-07


## 7. SHAP explanations, operations layer (Random Forest)

Plain `TreeExplainer` already works in probability space for a Random Forest. One-hot dummy
columns are summed back into their parent feature before ranking, and labels show the order's real
category value rather than the encoded column name.

In [12]:
ops_cells = cell_of(y_ops, (ops_proba >= OPS_THRESHOLD).astype(int))
ops_idx = np.concatenate([np.sort(rng.choice(np.where(ops_cells == c)[0], size=10, replace=False))
                          for c in ["TP", "FN", "FP", "TN"]])

ops_explainer = shap.TreeExplainer(rf_model)
raw_sv = ops_explainer.shap_values(X_ops.iloc[ops_idx])
ops_shap = raw_sv[1] if isinstance(raw_sv, list) else (raw_sv[:, :, 1] if np.ndim(raw_sv) == 3 else raw_sv)
ev = ops_explainer.expected_value
ops_base = float(ev[1]) if np.ndim(ev) else float(ev)
ops_recon = np.abs(ops_base + ops_shap.sum(axis=1) - ops_proba[ops_idx])
print(f"base value {ops_base:.6f}; reconciliation over {len(ops_idx)} cases: max diff {ops_recon.max():.2e}")
assert ops_recon.max() <= 1e-6

ONE_HOT_GROUPS = ["Type", "Customer Country", "Customer Segment", "Shipping Mode"]
groups = {}
for j, col in enumerate(OPS_FEATURES):
    g = next((g for g in ONE_HOT_GROUPS if col.startswith(f"low_cardinality__{g}_")), None)
    if g:
        groups.setdefault(g, []).append(j)
    else:
        groups[col.split("__", 1)[1]] = [j]

OPS_LABELS = {
    "Days for shipment (scheduled)": "Scheduled shipping window", "Order_Hour": "Order hour",
    "Order_DayOfWeek": "Order day of week", "Order_Year": "Order year", "Order_Month": "Order month",
    "GSCPI": "Supply chain pressure (GSCPI)", "Benefit per order": "Benefit per order", "Sales": "Sale amount",
    "Product Price": "Product price", "Order Item Quantity": "Order quantity", "Order Item Discount": "Order discount",
    "Order Item Discount Rate": "Discount rate", "Order Item Profit Ratio": "Profit ratio", "Type": "Payment type",
}
DOW = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def ops_display(feature, raw):
    v = raw[feature]
    if feature == "Days for shipment (scheduled)":
        return f"{int(v)} day{'s' if int(v) != 1 else ''}"
    if feature == "Order_DayOfWeek":
        return DOW[int(v)]
    if feature == "Order_Hour":
        return f"{int(v)}:00"
    if feature in ("Order_Year", "Order_Month", "Order Item Quantity"):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        return f"{float(v):.2f}"
    return str(v)

ops_cases = []
for k, i in enumerate(ops_idx):
    raw = X_ops_raw.iloc[i]
    grouped = [(g, float(ops_shap[k, idxs].sum())) for g, idxs in groups.items()]
    assert abs(sum(v for _, v in grouped) - ops_shap[k].sum()) < 1e-12
    grouped.sort(key=lambda t: abs(t[1]), reverse=True)
    top = grouped[:TOP_N]
    rest = float(ops_proba[i] - ops_base - sum(v for _, v in top))
    ops_cases.append({
        "id": f"OPS-T{i:05d}", "row": int(i), "cell": str(ops_cells[i]), "label": int(y_ops[i]),
        "proba": round(float(ops_proba[i]), 6), "band": band(ops_proba[i], OPS_THRESHOLD),
        "shipping_mode": str(raw["Shipping Mode"]), "scheduled_days": int(raw["Days for shipment (scheduled)"]),
        "market": str(raw["Market"]), "category": str(raw["Category Name"]), "segment": str(raw["Customer Segment"]),
        "drivers": [{"feature": g, "label": OPS_LABELS.get(g, g), "value": ops_display(g, raw), "contribution": round(v, 6)} for g, v in top],
        "rest": {"label": f"{len(groups) - TOP_N} more factors combined", "contribution": round(rest, 6)},
        "reconciliation_abs_diff": float(ops_recon[k]),
    })

with open(OUT / "operations_shap_cases.json", "w") as f:
    json.dump({"method": "shap.TreeExplainer (Random Forest leaf probabilities), one-hot dummies grouped to parent feature",
               "base_value": round(ops_base, 6), "threshold": OPS_THRESHOLD, "top_n": TOP_N,
               "grouped_feature_count": len(groups), "cases": ops_cases}, f, indent=1)
print(f"wrote operations_shap_cases.json: {len(ops_cases)} cases,", pd.Series([c["cell"] for c in ops_cases]).value_counts().to_dict())

base value 0.548184; reconciliation over 40 cases: max diff 4.00e-13
wrote operations_shap_cases.json: 40 cases, {'TP': 10, 'FN': 10, 'FP': 10, 'TN': 10}


## 8. Financial what-if model

The console can re-score a firm in the browser if the XGBoost trees and the scaler are exported as
JSON and a JavaScript scorer reproduces Python exactly. `save_raw("json")` serialises the booster
to memory; the `.pkl` file is not written. The parity test lives in `console/scripts/parity.mjs`.

In [13]:
booster_json = json.loads(xgb_model.get_booster().save_raw(raw_format="json"))
top6 = list(fin_importance.head(6).index)
bounds = {n: {"p01": float(X_fin_train_raw[n].quantile(0.01)), "p99": float(X_fin_train_raw[n].quantile(0.99)),
              "median": float(X_fin_train_raw[n].median())} for n in top6}
whatif = {
    "feature_names": list(fin_pre.feature_names_in_),
    "model_feature_order": FIN_FEATURES,
    "scaler_mean": [float(v) for v in fin_pre.mean_],
    "scaler_scale": [float(v) for v in fin_pre.scale_],
    "booster": booster_json,
    "top_features": [{"feature": n, "label": FIN_LABELS.get(n, n), **bounds[n]} for n in top6],
}
with open(OUT / "financial_whatif_model.json", "w") as f:
    json.dump(whatif, f, separators=(",", ":"))

# Parity inputs: raw rows and the Python probability at full precision, computed from the same
# raw -> scaler.transform -> predict_proba path the browser will use.
p_from_raw = xgb_model.predict_proba(pd.DataFrame(fin_pre.transform(X_fin_raw), columns=fin_pre.get_feature_names_out())[FIN_FEATURES])[:, 1]
with open(OUT / "financial_test_raw.json", "w") as f:
    json.dump({"feature_names": list(fin_pre.feature_names_in_), "rows": X_fin_raw.to_numpy().tolist(),
               "python_proba": [float(p) for p in p_from_raw], "label": [int(v) for v in y_fin]}, f, separators=(",", ":"))
print("top 6 features by mean |SHAP| (what-if sliders):", top6)
print("max |p_from_raw - p_from_processed| =", np.abs(p_from_raw - fin_proba).max())

top 6 features by mean |SHAP| (what-if sliders): ['Quick Ratio', 'Total debt/Total net worth', 'Persistent EPS in the Last Four Seasons', 'Retained Earnings to Total Assets', 'Profitability_Composite', 'Borrowing dependency']
max |p_from_raw - p_from_processed| = 0.0


## 9. Confirm the artifacts are unchanged and list outputs

In [14]:
for k, p in ARTIFACTS.items():
    assert sha256(p) == HASH_BEFORE[k], f"{k} changed on disk"
print("all four .pkl files unchanged (sha256 identical before and after)")
for p in sorted(OUT.glob("*.json")):
    print(f"{p.name:<44} {p.stat().st_size:>10,} bytes")

all four .pkl files unchanged (sha256 identical before and after)
dataset_facts.json                                  372 bytes
financial_shap_cases.json                        33,001 bytes
financial_test_raw.json                       2,092,973 bytes
financial_test_scores.json                       14,561 bytes
financial_whatif_model.json                     514,117 bytes
operations_profit_ratio_v2_transform.json           210 bytes
operations_shap_cases.json                       52,404 bytes
operations_test_scores.json                     649,296 bytes
